# Phase 8 — Model Selection & Multiple ML Algorithms

## H&M Personalized Fashion Recommendations → Customer Purchase Prediction

### Objective

In Phase 7, we established:

- a `DummyClassifier` benchmark,
- a Logistic Regression baseline,
- classification metrics,
- threshold analysis,
- train-vs-validation comparison,
- initial bias–variance analysis.

In Phase 8, we move from a single baseline model to a **systematic comparison of multiple machine-learning algorithms**.

The central question is:

> **Which model family captures future customer purchase behavior most effectively while maintaining good generalization?**

---

## Models covered

We will compare:

1. Dummy Classifier — reference only
2. Logistic Regression — linear baseline
3. Decision Tree — nonlinear single-tree model
4. Random Forest — bagging ensemble
5. Extra Trees — highly randomized tree ensemble
6. HistGradientBoosting — gradient boosting model
7. XGBoost — external gradient-boosting implementation, when installed

---

## Evaluation principles

Every model will be evaluated using the **same temporal validation framework** established in Phase 5.

Primary metrics:

- PR-AUC
- ROC-AUC
- F1
- Precision
- Recall
- Accuracy

Additional engineering metrics:

- Training time
- Prediction time
- Number of parameters / estimators where meaningful
- Train-validation generalization gap

> **The test set will not be used for model selection.**

The test set remains the final holdout and will be evaluated only after selecting the best candidate using the training and validation data.


# 8.1 Why Compare Multiple Model Families?

Customer purchase behavior is unlikely to be perfectly linear.

For example:

```text
Recent spending
        +
Purchase frequency
        +
Customer tenure
        +
Membership status
        ↓
Nonlinear purchase probability
```

A linear model assumes a relatively simple relationship between features and the log-odds of purchase.

Tree-based models can learn rules such as:

```text
IF recent_items is high
AND recency is low
AND customer is active
THEN purchase probability increases
```

Ensemble models can combine many such patterns.

Therefore, model comparison allows us to determine whether the additional complexity actually improves generalization.


# 8.2 Model Families

## 1. Logistic Regression

A linear baseline.

**Strengths**
- Fast
- Interpretable
- Works well with sparse one-hot features
- Strong benchmark

**Weakness**
- Limited nonlinear interaction modeling

---

## 2. Decision Tree

Learns nonlinear decision rules.

**Strengths**
- Easy to interpret
- Captures nonlinear relationships
- No scaling requirement

**Weaknesses**
- Can overfit easily
- High variance

---

## 3. Random Forest

An ensemble of decision trees trained using bagging and feature randomness.

**Strengths**
- Strong nonlinear baseline
- Usually more stable than a single tree
- Handles interactions naturally

**Weaknesses**
- Larger computational cost
- Can require substantial memory

---

## 4. Extra Trees

Extremely randomized trees.

Compared with Random Forest, Extra Trees introduce additional randomness in split selection.

**Strengths**
- Often strong on tabular data
- Can reduce variance
- Usually fast relative to heavily tuned boosting

---

## 5. HistGradientBoosting

A gradient boosting algorithm optimized for large tabular datasets.

**Strengths**
- Captures nonlinear relationships
- Efficient histogram-based training
- Strong tabular-data baseline

**Weakness**
- Does not naturally consume sparse one-hot matrices in the same way as linear models.

---

## 6. XGBoost

A powerful gradient-boosting implementation widely used for structured/tabular machine learning.

It will be included when the package is available.

**Strengths**
- Excellent nonlinear modeling
- Regularization
- Strong predictive performance

**Weakness**
- More hyperparameters
- More computationally expensive
- Requires careful tuning


# 8.3 Critical Preprocessing Decision

Different model families have different input requirements.

### Linear models

Logistic Regression works naturally with:

```text
Numerical → Impute → Scale
Categorical → Impute → One-Hot
```

This creates a sparse matrix.

### Tree ensembles

Tree models do not require standardization.

However, the important issue is not scaling — it is **sparse/dense representation and computational cost**.

For this phase:

- Logistic Regression uses the Phase 6 sparse representation.
- Random Forest / Extra Trees use a dense representation **only if it is safe for the actual dataset size**.
- HistGradientBoosting and XGBoost use a numerical representation designed for tabular tree learning.
- High-cardinality categorical variables are handled carefully.

> We do not blindly convert a very large sparse matrix into dense format.

The notebook therefore contains memory checks before dense conversion.


# 8.4 Imports and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import time
import json
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

RANDOM_STATE = 42

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train_phase5.parquet"
VAL_PATH = PROCESSED_DIR / "validation_phase5.parquet"
TEST_PATH = PROCESSED_DIR / "test_phase5.parquet"

STANDARD_PREPROCESSOR_PATH = (
    MODELS_DIR / "preprocessor_standard_phase6.joblib"
)

print("Project root:", PROJECT_ROOT.resolve())
print("Data directory:", PROCESSED_DIR.resolve())
print("Models directory:", MODELS_DIR.resolve())
print("Results directory:", RESULTS_DIR.resolve())


# 8.5 Load Phase 5 Data

In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
val_df = pd.read_parquet(VAL_PATH)
test_df = pd.read_parquet(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


# 8.6 Separate Features, Target, and Customer IDs

The customer identifier is retained for tracking predictions but is excluded from the ML feature matrix.

The target remains:

```text
1 → purchase during next 30 days
0 → no purchase during next 30 days
```


In [ ]:
TARGET_COL = "target"
ID_COL = "customer_id"

customer_id_train = train_df[ID_COL].copy()
customer_id_val = val_df[ID_COL].copy()
customer_id_test = test_df[ID_COL].copy()

X_train = train_df.drop(columns=[TARGET_COL, ID_COL])
X_val = val_df.drop(columns=[TARGET_COL, ID_COL])
X_test = test_df.drop(columns=[TARGET_COL, ID_COL])

y_train = train_df[TARGET_COL].astype("int8")
y_val = val_df[TARGET_COL].astype("int8")
y_test = test_df[TARGET_COL].astype("int8")

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


# 8.7 Verify Temporal Split Integrity

Phase 8 must preserve the temporal split.

No random train-test split is performed here.

The validation set represents a later period than the training set, and the test set represents the final future period.

This means model selection is based on **future-period generalization**, which is more realistic for this forecasting-style classification problem.


In [ ]:
print("Training target rate:", y_train.mean())
print("Validation target rate:", y_val.mean())
print("Test target rate:", y_test.mean())

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

assert list(X_train.columns) == list(X_val.columns)
assert list(X_train.columns) == list(X_test.columns)

print("\nTemporal split structure verified.")


# 8.8 Load Phase 6 Standard Preprocessor

The Phase 6 preprocessing pipeline performs:

- numerical median imputation,
- missing indicators,
- numerical standardization,
- categorical missing-value handling,
- one-hot encoding.

We reuse the fitted preprocessing pipeline.

It was fitted on the training data only.


In [ ]:
preprocessor = joblib.load(
    STANDARD_PREPROCESSOR_PATH
)

print("Loaded preprocessing pipeline:")
print(preprocessor)


In [ ]:
X_train_processed = preprocessor.transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed train:", X_train_processed.shape)
print("Processed validation:", X_val_processed.shape)
print("Processed test:", X_test_processed.shape)
print("Sparse:", sp.issparse(X_train_processed))


# 8.9 Matrix Memory Audit

The H&M dataset can create a very large feature matrix after one-hot encoding.

We therefore calculate an approximate dense memory requirement before attempting any sparse-to-dense conversion.

This is an important engineering safeguard.

```text
dense_memory ≈ rows × columns × bytes_per_value
```

For `float64`, each value requires approximately 8 bytes.


In [ ]:
def dense_memory_gb(matrix, dtype=np.float32):
    rows, cols = matrix.shape
    return rows * cols * np.dtype(dtype).itemsize / (1024 ** 3)

print(
    "Estimated dense float32 train memory:",
    f"{dense_memory_gb(X_train_processed):.2f} GB"
)

print(
    "Estimated dense float32 validation memory:",
    f"{dense_memory_gb(X_val_processed):.2f} GB"
)

print(
    "Estimated dense float32 test memory:",
    f"{dense_memory_gb(X_test_processed):.2f} GB"
)


# 8.10 Configure Dense Conversion Safety

Some scikit-learn tree algorithms work best with dense arrays.

However, converting a large one-hot matrix to dense can exhaust system memory.

We therefore use a configurable limit.

If the estimated memory exceeds the limit, the notebook will **not automatically densify the complete matrix**.

This is preferable to crashing the notebook.

For a large H&M run, the recommended workflow is:

- Logistic Regression on sparse features,
- specialized tree/boosting representation,
- or sampling for exploratory model comparison.



In [ ]:
MAX_DENSE_GB = 4.0

dense_is_safe = dense_memory_gb(
    X_train_processed,
    dtype=np.float32
) <= MAX_DENSE_GB

print("Dense conversion safe:", dense_is_safe)
print("Configured limit:", MAX_DENSE_GB, "GB")


# 8.11 Create a Dense Matrix When Safe

For tree models that require dense input, we convert the sparse matrix only when the configured memory limit allows it.

If it is not safe, we create a controlled sample for tree-model benchmarking.

### Why sampling?

The goal of this phase is **model family comparison**, not final model training.

The final model will be trained and tuned later using the best representation and full available data.

The sample must still be drawn from the training period only.


In [ ]:
TREE_SAMPLE_SIZE = 200_000

if dense_is_safe:
    X_train_tree = X_train_processed.astype(np.float32).toarray()
    X_val_tree = X_val_processed.astype(np.float32).toarray()

    y_train_tree = y_train.to_numpy()
    y_val_tree = y_val.to_numpy()

    tree_data_mode = "full"

else:
    rng = np.random.default_rng(RANDOM_STATE)

    sample_size = min(
        TREE_SAMPLE_SIZE,
        X_train_processed.shape[0]
    )

    sample_indices = rng.choice(
        X_train_processed.shape[0],
        size=sample_size,
        replace=False
    )

    X_train_tree = X_train_processed[
        sample_indices
    ].astype(np.float32).toarray()

    y_train_tree = y_train.iloc[
        sample_indices
    ].to_numpy()

    # Validation is retained in full if its dense representation
    # is within the configured limit.
    if dense_memory_gb(
        X_val_processed,
        dtype=np.float32
    ) <= MAX_DENSE_GB:
        X_val_tree = X_val_processed.astype(
            np.float32
        ).toarray()
        y_val_tree = y_val.to_numpy()
    else:
        val_sample_size = min(
            TREE_SAMPLE_SIZE,
            X_val_processed.shape[0]
        )

        val_indices = rng.choice(
            X_val_processed.shape[0],
            size=val_sample_size,
            replace=False
        )

        X_val_tree = X_val_processed[
            val_indices
        ].astype(np.float32).toarray()

        y_val_tree = y_val.iloc[
            val_indices
        ].to_numpy()

    tree_data_mode = "sampled"

print("Tree-model data mode:", tree_data_mode)
print("Tree train:", X_train_tree.shape)
print("Tree validation:", X_val_tree.shape)


# 8.12 Evaluation Function

Every model should be evaluated consistently.

The function calculates:

- Accuracy
- Precision
- Recall
- F1
- ROC-AUC
- PR-AUC
- confusion-matrix components

For imbalanced classification, **PR-AUC is especially important**.

The probability score is used for ROC-AUC and PR-AUC, while the class prediction is used for threshold-dependent metrics.


In [ ]:
def evaluate_model(
    model_name,
    split_name,
    y_true,
    y_pred,
    y_prob,
    train_time=None,
    predict_time=None,
    training_rows=None
):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "model": model_name,
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true,
            y_prob
        ),
        "pr_auc": average_precision_score(
            y_true,
            y_prob
        ),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "train_time_seconds": train_time,
        "prediction_time_seconds": predict_time,
        "training_rows": training_rows
    }


def fit_and_evaluate(
    model,
    model_name,
    X_tr,
    y_tr,
    X_va,
    y_va
):
    start = time.perf_counter()
    model.fit(X_tr, y_tr)
    train_time = time.perf_counter() - start

    start = time.perf_counter()
    train_prob = model.predict_proba(X_tr)[:, 1]
    train_pred = (train_prob >= 0.5).astype(int)
    train_prediction_time = time.perf_counter() - start

    start = time.perf_counter()
    val_prob = model.predict_proba(X_va)[:, 1]
    val_pred = (val_prob >= 0.5).astype(int)
    val_prediction_time = time.perf_counter() - start

    results = [
        evaluate_model(
            model_name,
            "train",
            y_tr,
            train_pred,
            train_prob,
            train_time=train_time,
            predict_time=train_prediction_time,
            training_rows=len(y_tr)
        ),
        evaluate_model(
            model_name,
            "validation",
            y_va,
            val_pred,
            val_prob,
            train_time=train_time,
            predict_time=val_prediction_time,
            training_rows=len(y_tr)
        )
    ]

    return model, pd.DataFrame(results)


# 8.13 Model 1 — Dummy Classifier

The DummyClassifier is retained as the reference point.

It does not use customer features.

This gives us the minimum benchmark that meaningful models should beat.


In [ ]:
dummy_model = DummyClassifier(
    strategy="prior"
)

dummy_model, dummy_results = fit_and_evaluate(
    dummy_model,
    "DummyClassifier",
    X_train_processed,
    y_train,
    X_val_processed,
    y_val
)

display(dummy_results)


# 8.14 Model 2 — Logistic Regression

This is the Phase 7 baseline and is included again so that the Phase 8 leaderboard contains all models in one place.

We use the sparse Phase 6 representation.


In [ ]:
logistic_model = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="liblinear",
    max_iter=1000,
    random_state=RANDOM_STATE
)

logistic_model, logistic_results = fit_and_evaluate(
    logistic_model,
    "LogisticRegression",
    X_train_processed,
    y_train,
    X_val_processed,
    y_val
)

display(logistic_results)


# 8.15 Model 3 — Decision Tree

A Decision Tree is our first nonlinear model.

We deliberately restrict its depth.

An unrestricted decision tree can memorize the training data, producing:

```text
Training score → very high
Validation score → much lower
```

That would be a classic high-variance model.

This first experiment therefore acts as a controlled nonlinear benchmark.


In [ ]:
decision_tree = DecisionTreeClassifier(
    max_depth=10,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

decision_tree, decision_tree_results = fit_and_evaluate(
    decision_tree,
    "DecisionTree",
    X_train_tree,
    y_train_tree,
    X_val_tree,
    y_val_tree
)

display(decision_tree_results)


# 8.16 Model 4 — Random Forest

Random Forest uses bagging:

```text
Training data
     │
     ├── Tree 1
     ├── Tree 2
     ├── Tree 3
     ├── ...
     └── Tree N
             ↓
       Aggregated prediction
```

Each tree sees a randomized version of the data/features.

This generally reduces the variance of an individual decision tree.


In [ ]:
random_forest = RandomForestClassifier(
    n_estimators=150,
    max_depth=14,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

random_forest, random_forest_results = fit_and_evaluate(
    random_forest,
    "RandomForest",
    X_train_tree,
    y_train_tree,
    X_val_tree,
    y_val_tree
)

display(random_forest_results)


# 8.17 Model 5 — Extra Trees

Extra Trees introduces additional randomization into tree construction.

Compared with Random Forest, split thresholds are randomized more aggressively.

This can produce a strong low-variance ensemble and is a useful comparison against Random Forest.


In [ ]:
extra_trees = ExtraTreesClassifier(
    n_estimators=150,
    max_depth=14,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

extra_trees, extra_trees_results = fit_and_evaluate(
    extra_trees,
    "ExtraTrees",
    X_train_tree,
    y_train_tree,
    X_val_tree,
    y_val_tree
)

display(extra_trees_results)


# 8.18 Model 6 — HistGradientBoosting

Gradient boosting builds models sequentially.

Conceptually:

```text
Model 1
  ↓
Residual errors
  ↓
Model 2 learns errors
  ↓
Residual errors
  ↓
Model 3 learns errors
  ↓
...
  ↓
Final ensemble
```

Each new tree attempts to improve the mistakes made by the previous ensemble.

`HistGradientBoostingClassifier` uses histogram-based learning and is designed for efficient tabular modeling.

Because it works with dense numerical arrays, it uses the controlled tree-model representation created above.


In [ ]:
hist_gradient_boosting = HistGradientBoostingClassifier(
    learning_rate=0.08,
    max_iter=150,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    random_state=RANDOM_STATE
)

hist_gradient_boosting, hist_gradient_results = fit_and_evaluate(
    hist_gradient_boosting,
    "HistGradientBoosting",
    X_train_tree,
    y_train_tree,
    X_val_tree,
    y_val_tree
)

display(hist_gradient_results)


# 8.19 Model 7 — XGBoost (Optional)

XGBoost is a powerful gradient-boosting implementation.

We use a small, regularized configuration for the first comparison.

If XGBoost is not installed, this section is skipped rather than breaking the complete notebook.

### Why start with conservative settings?

The goal of Phase 8 is **model-family comparison**, not exhaustive hyperparameter tuning.

Hyperparameter optimization belongs in a later phase.


In [ ]:
try:
    from xgboost import XGBClassifier
    xgboost_available = True
    print("XGBoost is available.")
except ImportError:
    xgboost_available = False
    print(
        "XGBoost is not installed. "
        "The XGBoost experiment will be skipped."
    )


In [ ]:
xgb_results = pd.DataFrame()
xgb_model = None

if xgboost_available:
    positive_rate = float(y_train_tree.mean())
    negative_rate = 1.0 - positive_rate

    scale_pos_weight = (
        negative_rate / positive_rate
        if positive_rate > 0
        else 1.0
    )

    xgb_model = XGBClassifier(
        n_estimators=250,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=10,
        reg_lambda=1.0,
        reg_alpha=0.0,
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        scale_pos_weight=scale_pos_weight
    )

    xgb_model, xgb_results = fit_and_evaluate(
        xgb_model,
        "XGBoost",
        X_train_tree,
        y_train_tree,
        X_val_tree,
        y_val_tree
    )

    display(xgb_results)
else:
    print("XGBoost experiment skipped.")


# 8.20 Combine All Model Results

We now create a single experiment table.

This table becomes the central Phase 8 leaderboard.

The most important comparison is between **validation metrics**, because training metrics can be inflated by overfitting.


In [ ]:
all_results = pd.concat(
    [
        dummy_results,
        logistic_results,
        decision_tree_results,
        random_forest_results,
        extra_trees_results,
        hist_gradient_results,
        xgb_results
    ],
    ignore_index=True
)

display(
    all_results[
        [
            "model",
            "split",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "train_time_seconds"
        ]
    ]
)


# 8.21 Validation Leaderboard

For this problem, PR-AUC is an important ranking metric because the positive purchase class may be relatively uncommon.

We therefore sort the validation leaderboard primarily by:

1. PR-AUC
2. F1
3. ROC-AUC

This is not a universal ranking rule. The final metric priority should reflect the business objective.

For this academic project, this gives us a sensible starting point.


In [ ]:
validation_leaderboard = (
    all_results[
        all_results["split"] == "validation"
    ]
    .sort_values(
        ["pr_auc", "f1", "roc_auc"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    validation_leaderboard[
        [
            "model",
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "train_time_seconds",
            "prediction_time_seconds"
        ]
    ]
)


# 8.22 Train vs Validation Generalization Gap

A high validation score alone is not enough.

We also need to determine whether the model is overfitting.

For each model:

\[
Generalization\ Gap =
Train\ Score - Validation\ Score
\]

A large positive gap can indicate high variance.

A small gap with weak performance can indicate underfitting or high bias.

We inspect the gap for F1, ROC-AUC and PR-AUC.


In [ ]:
train_results = all_results[
    all_results["split"] == "train"
].copy()

val_results = all_results[
    all_results["split"] == "validation"
].copy()

train_indexed = train_results.set_index("model")
val_indexed = val_results.set_index("model")

gap_rows = []

for model_name in val_indexed.index:
    if model_name not in train_indexed.index:
        continue

    gap_rows.append({
        "model": model_name,
        "train_f1": train_indexed.loc[
            model_name, "f1"
        ],
        "validation_f1": val_indexed.loc[
            model_name, "f1"
        ],
        "f1_gap": (
            train_indexed.loc[
                model_name, "f1"
            ]
            -
            val_indexed.loc[
                model_name, "f1"
            ]
        ),
        "train_roc_auc": train_indexed.loc[
            model_name, "roc_auc"
        ],
        "validation_roc_auc": val_indexed.loc[
            model_name, "roc_auc"
        ],
        "roc_auc_gap": (
            train_indexed.loc[
                model_name, "roc_auc"
            ]
            -
            val_indexed.loc[
                model_name, "roc_auc"
            ]
        ),
        "train_pr_auc": train_indexed.loc[
            model_name, "pr_auc"
        ],
        "validation_pr_auc": val_indexed.loc[
            model_name, "pr_auc"
        ],
        "pr_auc_gap": (
            train_indexed.loc[
                model_name, "pr_auc"
            ]
            -
            val_indexed.loc[
                model_name, "pr_auc"
            ]
        )
    })

gap_df = pd.DataFrame(gap_rows)

display(gap_df.sort_values("validation_pr_auc", ascending=False))


# 8.23 Visualize Validation Model Performance

A leaderboard is useful, but a visualization makes the differences easier to inspect.

We compare:

- Validation PR-AUC
- Validation ROC-AUC
- Validation F1


In [ ]:
plot_df = validation_leaderboard.copy()

models = plot_df["model"].tolist()
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(13, 7))

ax.bar(
    x - width,
    plot_df["pr_auc"],
    width,
    label="PR-AUC"
)

ax.bar(
    x,
    plot_df["roc_auc"],
    width,
    label="ROC-AUC"
)

ax.bar(
    x + width,
    plot_df["f1"],
    width,
    label="F1"
)

ax.set_xticks(x)
ax.set_xticklabels(models, rotation=35, ha="right")
ax.set_ylabel("Validation Score")
ax.set_ylim(0, 1)
ax.set_title("Phase 8 — Validation Model Comparison")
ax.legend()

plt.tight_layout()
plt.show()


# 8.24 Visualize Overfitting / Generalization Gap

We compare training and validation F1.

A model with:

```text
Train F1 >> Validation F1
```

is showing a potential high-variance problem.

A model with:

```text
Train F1 ≈ Validation F1
```

has a smaller generalization gap.

However, a small gap is not automatically good — both values could still be poor.


In [ ]:
gap_plot_df = gap_df.sort_values(
    "validation_f1",
    ascending=False
)

x = np.arange(len(gap_plot_df))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))

ax.bar(
    x - width / 2,
    gap_plot_df["train_f1"],
    width,
    label="Train F1"
)

ax.bar(
    x + width / 2,
    gap_plot_df["validation_f1"],
    width,
    label="Validation F1"
)

ax.set_xticks(x)
ax.set_xticklabels(
    gap_plot_df["model"],
    rotation=35,
    ha="right"
)

ax.set_ylabel("F1")
ax.set_ylim(0, 1)
ax.set_title("Train vs Validation F1")
ax.legend()

plt.tight_layout()
plt.show()


# 8.25 Model Complexity vs Performance

The purpose of model selection is not simply:

> Pick the most complex model.

Instead, we want:

> **The simplest model that provides strong generalization for the required objective.**

For example:

- If Logistic Regression performs almost as well as a complex ensemble, the simpler model may be preferable.
- If boosting substantially improves validation PR-AUC without a huge generalization gap, the additional complexity may be justified.
- If a Decision Tree achieves excellent training performance but weak validation performance, it is likely overfitting.

This reasoning is important for placement interviews because model selection is a trade-off between predictive power, complexity, interpretability and compute cost.


# 8.26 Feature Importance — Tree Models

Tree ensembles provide feature importance estimates.

For this first analysis we use impurity-based importance.

### Important caveat

Impurity importance can be biased toward:

- high-cardinality variables,
- frequently used split variables,
- correlated features.

Therefore, these importances are useful for **initial interpretation**, but they should not be treated as causal explanations.

Later, we can use permutation importance or SHAP for stronger model interpretation.


In [ ]:
def extract_tree_importance(model, feature_names, top_n=30):
    if not hasattr(model, "feature_importances_"):
        raise ValueError(
            "Model does not expose feature_importances_."
        )

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance": model.feature_importances_
    })

    return (
        importance_df
        .sort_values("importance", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

feature_names = preprocessor.get_feature_names_out()

print("Random Forest top features:")
display(
    extract_tree_importance(
        random_forest,
        feature_names,
        top_n=30
    )
)

print("Extra Trees top features:")
display(
    extract_tree_importance(
        extra_trees,
        feature_names,
        top_n=30
    )
)


# 8.27 Decision Tree Structure

A single Decision Tree is useful for visual intuition.

The model learns hierarchical rules.

For example, a conceptual tree might learn:

```text
Is recency < threshold?
       │
   ┌───┴───┐
   YES     NO
   │        │
 recent    less
 buyer     active
```

We intentionally keep the tree shallow enough to avoid an enormous visualization.


In [ ]:
print("Decision Tree depth:", decision_tree.get_depth())
print("Decision Tree leaves:", decision_tree.get_n_leaves())


# 8.28 Random Forest Complexity

We record basic model complexity statistics.

For tree ensembles:

- number of trees,
- maximum depth,
- number of features.

This helps document the experiment and compare computational trade-offs.


In [ ]:
complexity_rows = [
    {
        "model": "DecisionTree",
        "n_estimators": 1,
        "max_depth": decision_tree.get_depth(),
        "training_rows": len(y_train_tree)
    },
    {
        "model": "RandomForest",
        "n_estimators": random_forest.n_estimators,
        "max_depth": random_forest.max_depth,
        "training_rows": len(y_train_tree)
    },
    {
        "model": "ExtraTrees",
        "n_estimators": extra_trees.n_estimators,
        "max_depth": extra_trees.max_depth,
        "training_rows": len(y_train_tree)
    }
]

complexity_df = pd.DataFrame(complexity_rows)

if xgb_model is not None:
    complexity_df = pd.concat(
        [
            complexity_df,
            pd.DataFrame([{
                "model": "XGBoost",
                "n_estimators": xgb_model.n_estimators,
                "max_depth": xgb_model.max_depth,
                "training_rows": len(y_train_tree)
            }])
        ],
        ignore_index=True
    )

display(complexity_df)


# 8.29 Select the Best Phase 8 Candidate

The selection is based on validation performance, not test performance.

A practical selection strategy is:

1. Identify models with strong validation PR-AUC.
2. Check F1, precision and recall.
3. Check ROC-AUC.
4. Inspect train-validation gaps.
5. Consider training and inference cost.
6. Prefer a model with a good performance/complexity trade-off.

We do not automatically declare a winner using one metric without inspecting the complete picture.


In [ ]:
selection_table = validation_leaderboard[
    [
        "model",
        "pr_auc",
        "roc_auc",
        "f1",
        "precision",
        "recall",
        "accuracy",
        "train_time_seconds"
    ]
].copy()

selection_table = selection_table.merge(
    gap_df[
        [
            "model",
            "f1_gap",
            "roc_auc_gap",
            "pr_auc_gap"
        ]
    ],
    on="model",
    how="left"
)

display(selection_table)


# 8.30 Choose the Phase 8 Candidate

For the next phase, we choose the highest validation PR-AUC model as the **initial candidate**, subject to checking its generalization gap.

This is only a provisional selection.

The selected model will be subjected to:

- stronger temporal validation,
- hyperparameter tuning,
- regularization experiments,
- class-weight experiments,
- threshold optimization,
- final test evaluation.

Those experiments will happen later.


In [ ]:
best_model_name = validation_leaderboard.iloc[0]["model"]

best_model_row = validation_leaderboard.iloc[0]

print("Initial Phase 8 candidate:", best_model_name)
print("Validation PR-AUC:", best_model_row["pr_auc"])
print("Validation ROC-AUC:", best_model_row["roc_auc"])
print("Validation F1:", best_model_row["f1"])

candidate_gap = gap_df[
    gap_df["model"] == best_model_name
]

if not candidate_gap.empty:
    print(
        "PR-AUC train-validation gap:",
        float(candidate_gap["pr_auc_gap"].iloc[0])
    )


# 8.31 Save Model Comparison Results

The complete experiment table is saved so that later phases can reproduce the model-selection decision.

We save:

- all train/validation metrics,
- validation leaderboard,
- generalization gaps,
- model complexity information.


In [ ]:
ALL_RESULTS_PATH = RESULTS_DIR / "phase8_all_model_results.csv"
LEADERBOARD_PATH = RESULTS_DIR / "phase8_validation_leaderboard.csv"
GAP_PATH = RESULTS_DIR / "phase8_generalization_gaps.csv"
COMPLEXITY_PATH = RESULTS_DIR / "phase8_model_complexity.csv"

all_results.to_csv(
    ALL_RESULTS_PATH,
    index=False
)

validation_leaderboard.to_csv(
    LEADERBOARD_PATH,
    index=False
)

gap_df.to_csv(
    GAP_PATH,
    index=False
)

complexity_df.to_csv(
    COMPLEXITY_PATH,
    index=False
)

print("Saved:")
print(ALL_RESULTS_PATH)
print(LEADERBOARD_PATH)
print(GAP_PATH)
print(COMPLEXITY_PATH)


# 8.32 Save Candidate Models

We save the trained models that are useful for subsequent experimentation.

The Logistic Regression, Random Forest, Extra Trees, Decision Tree and HistGradientBoosting models are saved.

If XGBoost is available, it is also saved.

> The preprocessing pipeline remains a separate artifact from Phase 6.


In [ ]:
model_artifacts = {
    "logistic_regression_phase8": logistic_model,
    "decision_tree_phase8": decision_tree,
    "random_forest_phase8": random_forest,
    "extra_trees_phase8": extra_trees,
    "hist_gradient_boosting_phase8": hist_gradient_boosting
}

if xgb_model is not None:
    model_artifacts["xgboost_phase8"] = xgb_model

saved_paths = []

for artifact_name, model in model_artifacts.items():
    path = MODELS_DIR / f"{artifact_name}.joblib"
    joblib.dump(model, path)
    saved_paths.append(path)

for path in saved_paths:
    print("Saved:", path)


# 8.33 Save Phase 8 Experiment Configuration

Reproducibility requires documenting the important experiment settings.

We save:

- random seed,
- dense-memory limit,
- tree sample size,
- candidate model configurations,
- selected candidate name.


In [ ]:
phase8_config = {
    "random_state": RANDOM_STATE,
    "max_dense_gb": MAX_DENSE_GB,
    "tree_sample_size": TREE_SAMPLE_SIZE,
    "tree_data_mode": tree_data_mode,
    "selected_candidate": best_model_name,
    "models": {
        "logistic_regression": {
            "penalty": "l2",
            "C": 1.0,
            "solver": "liblinear",
            "max_iter": 1000
        },
        "decision_tree": {
            "max_depth": 10,
            "min_samples_leaf": 50,
            "class_weight": "balanced"
        },
        "random_forest": {
            "n_estimators": 150,
            "max_depth": 14,
            "min_samples_leaf": 20,
            "max_features": "sqrt"
        },
        "extra_trees": {
            "n_estimators": 150,
            "max_depth": 14,
            "min_samples_leaf": 20,
            "max_features": "sqrt"
        },
        "hist_gradient_boosting": {
            "learning_rate": 0.08,
            "max_iter": 150,
            "max_leaf_nodes": 31,
            "min_samples_leaf": 50
        }
    },
    "xgboost_available": xgboost_available
}

CONFIG_PATH = RESULTS_DIR / "phase8_experiment_config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(
        phase8_config,
        f,
        indent=4
    )

print("Saved:", CONFIG_PATH)


# 8.34 Final Phase 8 Interpretation Checklist

After executing this notebook, answer the following questions.

### Model performance

1. Which model has the highest validation PR-AUC?
2. Which model has the highest validation ROC-AUC?
3. Which model has the highest F1?
4. How much does the best model improve over Logistic Regression?

### Generalization

5. Which model has the largest train-validation gap?
6. Is the best model overfitting?
7. Does the Decision Tree behave like a high-variance learner?
8. Do Random Forest and Extra Trees reduce that variance?

### Model family comparison

9. Does nonlinear modeling improve performance?
10. Does boosting outperform bagging?
11. Does XGBoost provide meaningful improvement over simpler ensembles?
12. Is the improvement large enough to justify additional computational complexity?

### Business interpretation

13. Is precision or recall more important for this use case?
14. Is PR-AUC a better headline metric than accuracy?
15. Would a threshold different from 0.50 be useful?

---

# Important

Do **not** tune the models using the test set.

The test set should remain untouched until the final selected model has been determined.


# 8.35 Phase 8 Deliverables

After execution, the project should contain:

```text
models/
├── logistic_regression_phase8.joblib
├── decision_tree_phase8.joblib
├── random_forest_phase8.joblib
├── extra_trees_phase8.joblib
├── hist_gradient_boosting_phase8.joblib
└── xgboost_phase8.joblib          # if installed

results/
├── phase8_all_model_results.csv
├── phase8_validation_leaderboard.csv
├── phase8_generalization_gaps.csv
├── phase8_model_complexity.csv
└── phase8_experiment_config.json
```

---

# Phase 9 Preview — Optimization & Gradient Methods

Phase 8 tells us **which model families are promising**.

Phase 9 will investigate optimization and gradient-based learning more deeply.

Topics can include:

- Gradient Descent
- Stochastic Gradient Descent
- Mini-batch Gradient Descent
- Learning-rate experiments
- Convergence behavior
- Regularization
- Logistic Regression with `SGDClassifier`
- Loss-function comparison
- Batch-size effects
- Learning curves
- Optimization vs generalization

This will give the project a stronger **core machine-learning component** rather than simply being a collection of model API calls.
